In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, GAE
import pandas as pd
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
from torch_geometric.utils import from_scipy_sparse_matrix
from torch_geometric.data import Data
import scipy.sparse as sp
import networkx as nx
import matplotlib.pyplot as plt
from torch.nn import BatchNorm1d
from sklearn.metrics import roc_auc_score, average_precision_score, precision_score, recall_score, f1_score
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv
from torch.nn import BatchNorm1d
from torch_geometric.utils import dropout_adj
import random
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)


In [3]:
# Functions
class GCNEncoder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers=3, dropout_rate=0.01, edge_dropout_rate=0.1):
        super(GCNEncoder, self).__init__()
        self.num_layers = num_layers
        self.dropout_rate = dropout_rate
        self.edge_dropout_rate = edge_dropout_rate
        
        # First layer: input to 2x hidden size
        self.convs = torch.nn.ModuleList()
        self.convs.append(GCNConv(in_channels, 2 * hidden_channels, cached=False))
        
        # Add intermediate layers
        for _ in range(num_layers - 2):
            self.convs.append(GCNConv(2 * hidden_channels, 2 * hidden_channels, cached=False))
        
        # Final layer: hidden size to output size
        self.convs.append(GCNConv(2 * hidden_channels, out_channels, cached=False))
    
        # Batch normalization layers for each convolution layer
        #self.bns = torch.nn.ModuleList([BatchNorm1d(2 * hidden_channels) for _ in range(num_layers - 1)])
        self.bns = torch.nn.ModuleList([BatchNorm1d(2 * hidden_channels, momentum=0.1) for _ in range(num_layers - 1)])

    
        # Dropout rate
        self.dropout = torch.nn.Dropout(p=dropout_rate)
    
        # LeakyReLU instead of ReLU
        #self.activation = torch.nn.LeakyReLU(negative_slope=0.05)
        #self.activation = torch.nn.Sigmoid()
        #self.activation_hidden = torch.nn.LeakyReLU(negative_slope=0.05)  # For hidden layers
        self.activation_hidden = torch.nn.ReLU  # For hidden layers
        self.activation_output = torch.nn.Sigmoid()  # For final layer


    
    def forward(self, x, edge_index):
        # Apply edge dropout
        #edge_index, _ = dropout_adj(edge_index, p=self.edge_dropout_rate, training=self.training)
        
        # First layer (without batch normalization)
        x = self.convs[0](x, edge_index)
        #x = self.activation_hidden(x)
        x = F.relu(x)
    
    # Intermediate layers with batch normalization and dropout
        for i in range(1, self.num_layers - 1):
            x_residual = x # For residual connection
            x = self.convs[i](x, edge_index)
            x = self.bns[i-1](x) # Apply batch normalization
            #x = self.activation_hidden(x)
            x = F.relu(x)
            x = self.dropout(x)
            
            # Add residual connection (skip connection)
            x = x + x_residual
    
        # Final layer (no activation or batch norm)
        x = self.convs[-1](x, edge_index)
        x = F.relu(x)
        #x = self.activation_output(x)
        return x

In [4]:
# Normalize the adjacency matrix
def normalize_adj(adj):
    """Symmetrically normalize adjacency matrix."""
    adj = sp.coo_matrix(adj)
    rowsum = np.array(adj.sum(1)).flatten()
    d_inv_sqrt = np.power(rowsum + 1e-5, -0.5)
    d_inv_sqrt[np.isinf(d_inv_sqrt)] = 0.0
    d_mat_inv_sqrt = sp.diags(d_inv_sqrt)
    return adj.dot(d_mat_inv_sqrt).transpose().dot(d_mat_inv_sqrt).tocoo()

#normalized_laplacian
def normalize_adj(adj):
    """Compute the normalized Laplacian."""
    adj = sp.coo_matrix(adj)
    rowsum = np.array(adj.sum(1)).flatten()
    d_inv_sqrt = np.power(rowsum + 1e-5, -0.5)
    d_inv_sqrt[np.isinf(d_inv_sqrt)] = 0.0
    d_mat_inv_sqrt = sp.diags(d_inv_sqrt)
    identity = sp.eye(adj.shape[0])  # Identity matrix of the same size as adj
    laplacian = identity - d_mat_inv_sqrt.dot(adj).dot(d_mat_inv_sqrt)  # Normalized Laplacian
    return laplacian.tocoo()


def normalize_adj(adj):
    adj = adj + sp.eye(adj.shape[0])  # Add self-loops only
    return adj.tocoo()

In [5]:
# GAE model with weight decay and dropout
def apply_gae_to_adjacency(adj_matrix, node_features, embedding_dim=8, epochs=200, dropout_rate=0.01, edge_dropout_rate=0.8,weight_decay=1e-4):
    # Convert adjacency matrix to sparse format
    adj_sparse = sp.csr_matrix(adj_matrix)
    # Normalize the adjacency matrix
    adj_normalized = normalize_adj(adj_sparse + sp.eye(adj_sparse.shape[0]))
    # Get edge_index and edge_weight
    edge_index, edge_weight = from_scipy_sparse_matrix(adj_normalized)
    # Create data object
    x = torch.tensor(node_features, dtype=torch.float)
    data = Data(x=x, edge_index=edge_index)

    # Define the GCNEncoder as the encoder for the GAE model
    #encoder = GCNEncoder(in_channels=x.shape[1], hidden_channels=embedding_dim, out_channels=embedding_dim, dropout_rate=dropout_rate)
    encoder = GCNEncoder(in_channels=x.shape[1], hidden_channels=embedding_dim, out_channels=embedding_dim, dropout_rate=dropout_rate, edge_dropout_rate = edge_dropout_rate)
    #encoder = GCNEncoder(in_channels=x.shape[1], out_channels=embedding_dim, dropout_rate=dropout_rate)

    # Instantiate the GAE model by passing the encoder
    model = GAE(encoder)

    # Training settings
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    data = data.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=weight_decay)  # Applying weight decay le = 0.05

    # Training loop
    model.train()
    for epoch in range(1, epochs + 1):
        optimizer.zero_grad()
        z = model.encode(data.x, data.edge_index)
        loss = model.recon_loss(z, data.edge_index)
        loss.backward()
        optimizer.step()

    # After training, reconstruct adjacency matrix
    model.eval()
    with torch.no_grad():
        z = model.encode(data.x, data.edge_index)
        adj_reconstructed = model.decoder.forward_all(z).cpu().numpy()
        
    return adj_reconstructed

In [6]:
# Define the function to perform the GAE process for each threshold and multiple iterations
def run_gae_multiple_thresholds(data, ground_truth, algorithms, thresholds, num_iterations=30):
    num_genes = data.shape[0]
    #node_features = np.identity(num_genes)
    all_results = {}
    iteration_sums = {}

    for threshold in thresholds:
        print(f"Running for threshold: {threshold}")
        
        # Create a dictionary to store the results for each algorithm and threshold
        all_results[f'threshold_{threshold}'] = {}
        iteration_sums[f'threshold_{threshold}'] = {}

        # Initialize the iteration sum matrices for each threshold
        for iter_num in range(num_iterations):
            iteration_sums[f'threshold_{threshold}'][f'iter{iter_num + 1}_GCN_sum'] = np.zeros((num_genes, num_genes))

        for alg in algorithms:
            print(f"Running GAE for algorithm: {alg} with threshold: {threshold}")
            alg_columns = [col for col in data.columns if col.startswith(alg + '.')]
            alg_matrix = data[alg_columns].values

            # For each algorithm, create a list to store multiple runs
            all_results[f'threshold_{threshold}'][alg] = []

            for iteration in range(num_iterations):
                print(f"Iteration {iteration + 1}/{num_iterations} for algorithm: {alg}")
                
                # Apply GAE to the adjacency matrix of the algorithm
                adj_reconstructed = apply_gae_to_adjacency(alg_matrix, node_features)
                
                # Binarize the reconstructed adjacency matrix using the current threshold
                adj_binary = (adj_reconstructed > threshold).astype(int)

                # Store the binary adjacency matrix for this run
                all_results[f'threshold_{threshold}'][alg].append(adj_binary)

                # Add the binary matrix to the sum for this iteration and threshold
                iteration_sums[f'threshold_{threshold}'][f'iter{iteration + 1}_GCN_sum'] += adj_binary

    return all_results, iteration_sums

In [7]:
# Function to apply a second binarization threshold to the summed adjacency matrices
def apply_iteration_threshold(iteration_sums, thresholds, iter_thresholds):
    binary_sums = {}
    
    for threshold in thresholds:
        binary_sums[f'threshold_{threshold}'] = {}
        for iter_num in iteration_sums[f'threshold_{threshold}'].keys():
            binary_sums[f'threshold_{threshold}'][iter_num] = {}

            for iter_threshold in iter_thresholds:
                print(f"Applying iter_threshold {iter_threshold} to {iter_num} for threshold {threshold}")
                
                # Apply binarization on the summed adjacency matrix
                summed_matrix = iteration_sums[f'threshold_{threshold}'][iter_num]
                binary_matrix = (summed_matrix >= iter_threshold).astype(int)

                # Store the result
                binary_sums[f'threshold_{threshold}'][iter_num][f'iter_threshold_{iter_threshold}'] = binary_matrix
    
    return binary_sums

# Function to sum across iterations and apply a common threshold
def sum_across_iterations_and_binarize(binary_summed_matrices, thresholds, iter_thresholds, common_threshold):
    common_sums = {}

    for threshold in thresholds:
        common_sums[f'threshold_{threshold}'] = {}
        for iter_threshold in iter_thresholds:
            print(f"Summing across iterations for threshold {threshold} and iter_threshold {iter_threshold}")

            # Initialize the matrix for summing across iterations
            iteration_sum_matrix = np.zeros_like(binary_summed_matrices[f'threshold_{threshold}']['iter1_GCN_sum'][f'iter_threshold_{iter_threshold}'])

            # Sum the matrices across all iterations
            for iter_num in binary_summed_matrices[f'threshold_{threshold}'].keys():
                iteration_sum_matrix += binary_summed_matrices[f'threshold_{threshold}'][iter_num][f'iter_threshold_{iter_threshold}']

            # Apply the common threshold: if sum >= common_threshold, set 1; else set 0
            final_binarized_matrix = (iteration_sum_matrix >= common_threshold).astype(int)

            # Store the result
            common_sums[f'threshold_{threshold}'][f'iter_threshold_{iter_threshold}_common_threshold_{common_threshold}'] = final_binarized_matrix

    return common_sums


In [14]:
# Convert ground truth matrix to edge list
def convert_matrix_to_edge_list(matrix):
    edges = []
    for source in matrix.index:
        for target in matrix.columns:
            if matrix.at[source, target] == 1:
                edges.append((source, target))
    return edges

# Function to extract gene interactions (edges) from a binarized matrix
def extract_gene_interactions(final_matrix, gene_names):
    edges = []
    num_genes = final_matrix.shape[0]

    # Loop through the upper triangle of the matrix to avoid duplicates (since it's undirected)
    for i in range(num_genes):
        for j in range(i + 1, num_genes):
            if final_matrix[i, j] == 1:  # Check if there's an interaction
                edges.append(f"{gene_names[i]}-{gene_names[j]}")
    
    return edges


# Function to convert edges to consistent tuple format (e.g., ('G1', 'G10'))
def convert_edges_to_tuples(edge_list):
    edge_tuples = []
    for edge in edge_list:
        gene1, gene2 = edge.split('-')
        # Sort genes to maintain consistency in the edge representation (G1, G10) or (G10, G1)
        edge_tuples.append(tuple(sorted([gene1, gene2])))
    return set(edge_tuples)

# Function to compare predicted edges with known ground truth and generate the metrics
def compare_with_ground_truth(predicted_edges, known_edges_set):
    predicted_set = convert_edges_to_tuples(predicted_edges)
    
    # Calculate true positives and false positives
    true_positives = predicted_set.intersection(known_edges_set)
    false_positives = predicted_set.difference(known_edges_set)
    
    # Metrics
    num_true_positives = len(true_positives)
    num_false_positives = len(false_positives)
    
    return true_positives, false_positives, num_true_positives, num_false_positives

In [16]:
# Load the data
data = pd.read_csv("/Users/nirwantandukar/input_data/Database.DREAM4/NORMALIZED_EXP/normalized_weight_matrix_all_GRN_100_01_0.2filter.csv", index_col=0)
#data = pd.read_csv("/Users/nirwantandukar/input_data/SAP_lipids/NORMALIZED_EXP/normalized_weight_matrix_SAP_lowinput_0.2filter.csv", index_col=0)
#data = pd.read_csv("/Users/nirwantandukar/input_data/PSU_P/NORMALIZED_EXP/normalized_weight_matrix_B73_MICH_P_null_0.2filter.csv", index_col=0)
#data = pd.read_csv("/Users/nirwantandukar/input_data/weights_normalized/normalize_weight_matrix_all_GRN_100_01.csv", index_col=0)
print(f"Loaded data with shape: {data.shape}")

# Get the list of algorithms
columns = data.columns
algorithms = sorted(set(col.split('.')[0] for col in columns))
print(f"Algorithms found: {algorithms}")

num_genes = data.shape[0]
gene_names = data.index.tolist()
print(f"Number of genes: {num_genes}")

# Node features (identity matrix)
node_features = np.identity(num_genes)
print(f"Node features shape: {node_features.shape}")

# Load the gene expression data
#gene_expression_path = "/Users/nirwantandukar/input_data/Database.DREAM4/EXP/dream4_100_01_exp.csv"
#gene_expression_data = pd.read_csv(gene_expression_path, index_col=0)

# Convert the gene expression data to a numpy array (n x m matrix where n = genes and m = conditions/timepoints)
#node_features = gene_expression_data.values  # Use the expression data as node features
#print(np.size(node_features,1))

# Step 1: Load known (ground truth) interactions
file_path = "/Users/nirwantandukar/input_data/Database.DREAM4/GS/dream4_100_01_gs.csv"
ground_truth = pd.read_csv(file_path, index_col=0)
print(f"Ground truth loaded with shape: {ground_truth.shape}")

# Get the ground truth edges
known_edges_set = set(convert_matrix_to_edge_list(ground_truth))
print(f"Number of known edges: {len(known_edges_set)}")


Loaded data with shape: (100, 1700)
Algorithms found: ['GRN_ARACNE', 'GRN_BC3NET', 'GRN_C3NET', 'GRN_CLR', 'GRN_GENIE3_ET', 'GRN_GENIE3_RF', 'GRN_GRENIST', 'GRN_GRNBOOST2', 'GRN_INFERELATOR', 'GRN_KBOOST', 'GRN_LEAP', 'GRN_MEOMI', 'GRN_MRNET', 'GRN_MRNETB', 'GRN_NONLINEARODES', 'GRN_PCIT', 'GRN_TIGRESS']
Number of genes: 100
Node features shape: (100, 100)
Ground truth loaded with shape: (100, 100)
Number of known edges: 175


In [18]:
# Define thresholds and number of iterations
thresholds = [0.6]  # Different thresholds to run
#thresholds = [0.7,0.8,0.9]  # Different thresholds to run
num_iterations = 10  # Number of runs for each threshold

# Run the GAE process and store results for each threshold and algorithm
aggregated_results, iteration_sums = run_gae_multiple_thresholds(data, ground_truth, algorithms, thresholds, num_iterations)

# Define the additional binarization thresholds for the summed matrices
iter_thresholds = [2,4,5,6,7, 8,9,10,11,12,13,14]  # Example thresholds for binarizing summed matrices
#iter_thresholds = [4]  # Example thresholds for binarizing summed matrices

# Apply the iteration-level thresholding
binary_summed_matrices = apply_iteration_threshold(iteration_sums, thresholds, iter_thresholds)

# Example: Access binarized summed matrices for a specific iteration and iter_threshold
for threshold in thresholds:
    for iter_num in iteration_sums[f'threshold_{threshold}'].keys():
        for iter_threshold in iter_thresholds:
            binarized_matrix = binary_summed_matrices[f'threshold_{threshold}'][iter_num][f'iter_threshold_{iter_threshold}']
            #print(f"Binarized adjacency matrix for {iter_num}, threshold {threshold}, iter_threshold {iter_threshold}:")
            #print(binarized_matrix)

            # You can store or analyze these matrices further
            #output_file = f'binarized_{iter_num}_{threshold}_iter_thresh_{iter_threshold}.csv'
            #pd.DataFrame(binarized_matrix).to_csv(output_file)


# Define the common threshold
common_threshold = 5  # This is the threshold where if sum >= 30 across iterations, we set the value to 1. Consistently found in many iterations

# Sum across iterations and apply the common threshold
final_common_binarized_matrices = sum_across_iterations_and_binarize(binary_summed_matrices, thresholds, iter_thresholds, common_threshold)

# Example: Access and print the final binarized matrix for a specific threshold and iteration threshold
for threshold in thresholds:
    for iter_threshold in iter_thresholds:
        final_matrix = final_common_binarized_matrices[f'threshold_{threshold}'][f'iter_threshold_{iter_threshold}_common_threshold_{common_threshold}']
        #print(f"Final binarized matrix for threshold {threshold}, iter_threshold {iter_threshold}, common_threshold {common_threshold}:")
        #print(final_matrix)

        # You can save these matrices to CSV or further analyze them
        #output_file = f'final_binarized_threshold_{threshold}_iter_thresh_{iter_threshold}_common_thresh_{common_threshold}.csv'
        #pd.DataFrame(final_matrix).to_csv(output_file)


Running for threshold: 0.6
Running GAE for algorithm: GRN_ARACNE with threshold: 0.6
Iteration 1/10 for algorithm: GRN_ARACNE
Iteration 2/10 for algorithm: GRN_ARACNE
Iteration 3/10 for algorithm: GRN_ARACNE
Iteration 4/10 for algorithm: GRN_ARACNE
Iteration 5/10 for algorithm: GRN_ARACNE
Iteration 6/10 for algorithm: GRN_ARACNE
Iteration 7/10 for algorithm: GRN_ARACNE
Iteration 8/10 for algorithm: GRN_ARACNE
Iteration 9/10 for algorithm: GRN_ARACNE
Iteration 10/10 for algorithm: GRN_ARACNE
Running GAE for algorithm: GRN_BC3NET with threshold: 0.6
Iteration 1/10 for algorithm: GRN_BC3NET
Iteration 2/10 for algorithm: GRN_BC3NET
Iteration 3/10 for algorithm: GRN_BC3NET
Iteration 4/10 for algorithm: GRN_BC3NET
Iteration 5/10 for algorithm: GRN_BC3NET
Iteration 6/10 for algorithm: GRN_BC3NET
Iteration 7/10 for algorithm: GRN_BC3NET
Iteration 8/10 for algorithm: GRN_BC3NET
Iteration 9/10 for algorithm: GRN_BC3NET
Iteration 10/10 for algorithm: GRN_BC3NET
Running GAE for algorithm: GRN_C3NE

In [38]:
# Assuming the gene names are the row/column names from your original data
gene_names = list(data.index)  # Replace 'data' with your actual DataFrame that contains the gene names

# Example: Use the final binarized matrix from the previous step (e.g., for threshold 0.8, iter_threshold 6, common_threshold 2)
for threshold in thresholds:
    for iter_threshold in iter_thresholds:
        final_matrix = final_common_binarized_matrices[f'threshold_{threshold}'][f'iter_threshold_{iter_threshold}_common_threshold_{common_threshold}']
        
        # Extract gene interactions (edges)
        edges = extract_gene_interactions(final_matrix, gene_names)
        
        # Print the edges
        #print(f"Gene interactions for threshold {threshold}, iter_threshold {iter_threshold}, common_threshold {common_threshold}:")
        #for edge in edges:
         #   print(edge)

        # Optionally, save the edges to a file
        #output_file = f'gene_interactions_threshold_{threshold}_iter_thresh_{iter_threshold}_common_thresh_{common_threshold}.txt'
        #with open(output_file, 'w') as f:
        #    for edge in edges:
        #        f.write(edge + '\n')

# Example structure to store the results
results = []

# Assume thresholds, iter_thresholds, and common_threshold are defined (as in the previous steps)
for threshold in thresholds:
    for iter_threshold in iter_thresholds:
        # For each final binarized matrix (e.g., threshold 0.8, iter_threshold 6, common_threshold 2)
        final_matrix = final_common_binarized_matrices[f'threshold_{threshold}'][f'iter_threshold_{iter_threshold}_common_threshold_{common_threshold}']
        
        # Extract gene interactions (predicted edges) from the final matrix
        predicted_edges = extract_gene_interactions(final_matrix, gene_names)
        
        # Compare predicted edges with the known ground truth edges
        true_positives, false_positives, num_true_positives, num_false_positives = compare_with_ground_truth(predicted_edges, known_edges_set)
        
        # Create a row for the results table
        results.append({
            'Iterations': num_iterations,
            'Common_Iterations': common_threshold,
            'Iter_Threshold': iter_threshold,
            'Individual_Threshold': threshold,
            'True_Positives': ','.join([f"{tp[0]}-{tp[1]}" for tp in true_positives]),  # Format as G1-G10
            'False_Positives': ','.join([f"{fp[0]}-{fp[1]}" for fp in false_positives]),  # Format as G1-G10
            'No_True_Positives': num_true_positives,
            'No_False_Positives': num_false_positives
        })

# Convert the results to a DataFrame for better visualization
results_df = pd.DataFrame(results)

# Display the results
print(results_df)

# Optionally, save the results to a CSV file
results_df.to_csv('true_false_positives_summary_B73_MICH_P_null.csv', index=False)


    Iterations  Common_Iterations  Iter_Threshold  Individual_Threshold  \
0           10                  5               2                   0.6   
1           10                  5               4                   0.6   
2           10                  5               5                   0.6   
3           10                  5               6                   0.6   
4           10                  5               7                   0.6   
5           10                  5               8                   0.6   
6           10                  5               9                   0.6   
7           10                  5              10                   0.6   
8           10                  5              11                   0.6   
9           10                  5              12                   0.6   
10          10                  5              13                   0.6   
11          10                  5              14                   0.6   

                        

In [40]:
print(results_df)

    Iterations  Common_Iterations  Iter_Threshold  Individual_Threshold  \
0           10                  5               2                   0.6   
1           10                  5               4                   0.6   
2           10                  5               5                   0.6   
3           10                  5               6                   0.6   
4           10                  5               7                   0.6   
5           10                  5               8                   0.6   
6           10                  5               9                   0.6   
7           10                  5              10                   0.6   
8           10                  5              11                   0.6   
9           10                  5              12                   0.6   
10          10                  5              13                   0.6   
11          10                  5              14                   0.6   

                        